# testing s2

One reply on the current board. The agent stays where it is. Sampling and the stop rule match the learn-to-look beginnings generation: the trainer's `vl.generate` sampling, then eos, with a cap of `BEGINNING_MAX_NEW_TOKENS`.

The **Snapshot** dropdown lists `localizer/...` and `s2/...`: every directory under `weights/localizer/` or `weights/s2/` that `GemmaS2.load_snapshot` can read (`localizer.pt` plus an adapter, a `gemma/` bundle, or `train_meta.json` naming `gemma_base`), newest `localizer.pt` first. **Refresh** rescans. **Load** stacks that adapter on base Gemma and reads the Localizer. **New board** deals another training board (`n_gold` uniform in 0–3, opening `any`) and drops the previous tokens.

The teal card is the same game-settings editor as testing s1. **Generate** waits until **Render game settings** has written the form onto the board.

**Standard** is `standard_user_questions()`: the beginning line, then every look, then every move, for the corner phrases, the gold-region phrases, "the gold", and "the exit", then the two "your target" lines. Choosing one fills the question box. It does not run the model. **Generate** sends the text in the box.

**Prompt** chooses which trainer text wraps that question. Both modes use the trainer's builders, so the user turn is a notepad block, the picture, and a `Question / instruction:` line. The trace mode may also prepend a Memory-context block.

- **prompt at start of game** — `beginning_messages` with an empty session notepad (`format_notepad([])`) and the beginning question. The question box is not used.
- **prompt with fake NAMS trace and target selected** — `synthetic_prompt` on this board. It may write a fake `target` note and 0-3 earlier rounds about this board's golds, exits, and corners. The question box is the question, verbatim.

The picture in the prompt is the clean engine frame on the left. Training writes a noised copy of its frame; the dots are painted on the pixels this reply saw, and only in the widget.

After the reply finishes, each token is a click. A click draws that token's **s** in blue, **v** in red, **v_bar** in orange, and the average of **v** and **v_bar** in green. A point outside the unit square is left off the picture; the readout still prints the number.

**show all** is off until checked. It draws every token's four points at once. A clicked token is drawn larger on top of that.

**Clear** wipes the log, the tokens, and the dots.

Run this on the GPU box. The cell needs the repo environment (`pygame`, `torch`, `ipywidgets`).


In [ ]:
import html
import io
import math
import os
import random
import sys
import tempfile
import time
import traceback

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

# Run from the repo root so `agent` / `neural_net` imports resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import ipywidgets as widgets
import numpy as np
import torch
from IPython.display import HTML, display
from PIL import Image

from agent.game_io import (
    apply_edited_settings_dict,
    game_to_settings_dict,
    new_multi_gold_game,
)
from agent.memory import format_notepad

from agent.modes import (
    S2_BEGINNING_USER,
)
from agent.notebook_ui import settings_editor
from neural_net.loop import _eos_ids, _sample
from neural_net.paths import weights_root
from neural_net.render import canonical_frame
from neural_net.s2 import GemmaS2
from neural_net.s2_pretraining_learn_to_look import (
    BEGINNING_MAX_NEW_TOKENS,
    beginning_messages,
    standard_user_questions,
    synthetic_prompt,
)

BOARD_PX = 560
_RADIUS_ONE = 8
_RADIUS_ALL = 3
_COLOR = {
    "s": (30, 90, 255),
    "v": (255, 0, 0),
    "v_bar": (255, 140, 0),
    "avg": (0, 200, 70),
}
_DRAW_ORDER = ("s", "v_bar", "v", "avg")

# One reply's tokens. Each row is the decode step that produced it:
# id, piece, eos, s, v, v_bar, avg (v and v_bar).
tokens = []

state = {
    "busy": False,
    "ignore_through": 0,
    "gen": 0,
    "s2": None,
    "game": None,
    "frame": None,
    "selected": None,
}


def _snapshots():
    """Localizer directories under ``weights/localizer`` and ``weights/s2``, newest first.

    A snapshot ``load_snapshot`` can read has ``localizer.pt`` plus any of
    ``adapter_config.json``, ``gemma/gemma_meta.json``, or a
    ``train_meta.json`` that names ``gemma_base``.
    """
    import json

    found = []
    for folder in ("localizer", "s2"):
        root = weights_root() / folder
        if not root.is_dir():
            continue
        for head in root.rglob("localizer.pt"):
            if not head.is_file():
                continue
            parent = head.parent
            adapter = (parent / "adapter_config.json").is_file()
            bundled = (parent / "gemma" / "gemma_meta.json").is_file()
            named = False
            meta_path = parent / "train_meta.json"
            if meta_path.is_file():
                meta = json.loads(meta_path.read_text(encoding="utf-8"))
                named = bool(meta.get("gemma_base"))
            if adapter or bundled or named:
                found.append(parent)
    found.sort(
        key=lambda path: (
            (path / "localizer.pt").stat().st_mtime,
            path.as_posix(),
        ),
        reverse=True,
    )
    return found


def _snap_options():
    root = weights_root()
    found = _snapshots()
    if not found:
        return [("(no localizer snapshots)", "")]
    options = []
    for path in found:
        try:
            label = path.relative_to(root).as_posix()
        except ValueError:
            label = path.as_posix()
        options.append((label, str(path)))
    return options


def _question_options():
    options = [("(custom)", "")]
    for text in standard_user_questions():
        options.append((text, text))
    return options


def _png(rgb):
    buf = io.BytesIO()
    Image.fromarray(rgb, mode="RGB").save(buf, format="PNG")
    return buf.getvalue()


def _on_board(x, y):
    return (
        math.isfinite(x) and math.isfinite(y)
        and 0.0 <= x <= 1.0 and 0.0 <= y <= 1.0
    )


def _blit_dot(out, x, y, color, radius):
    if not _on_board(x, y):
        return
    height, width = out.shape[:2]
    col = int(round(float(x) * (width - 1)))
    row = int(round((1.0 - float(y)) * (height - 1)))
    y0 = max(0, row - radius)
    y1 = min(height, row + radius + 1)
    x0 = max(0, col - radius)
    x1 = min(width, col + radius + 1)
    yy, xx = np.ogrid[y0:y1, x0:x1]
    mask = (yy - row) ** 2 + (xx - col) ** 2 <= radius ** 2
    out[y0:y1, x0:x1][mask] = color


def _paint_layer(out, rows, radius):
    for key in _DRAW_ORDER:
        color = _COLOR[key]
        for row in rows:
            x, y = row[key]
            _blit_dot(out, x, y, color, radius)


def _overlay(frame):
    selected = state["selected"]
    show = bool(show_all.value) and bool(tokens)
    one = (
        tokens[selected]
        if selected is not None and 0 <= selected < len(tokens)
        else None
    )
    if not show and one is None:
        return frame
    out = np.array(frame, copy=True)
    if show:
        _paint_layer(out, tokens, _RADIUS_ALL)
    if one is not None:
        _paint_layer(out, [one], _RADIUS_ONE)
    return out


def _show_board():
    frame = state["frame"]
    if frame is None:
        return
    board.value = _png(_overlay(frame))


def _visible(piece):
    if piece == "":
        return "∅"
    return piece.replace(" ", "\u00a0")


def _token_markup():
    if not tokens:
        return "<div class='s2-tokens'>Generate to get clickable tokens.</div>"
    gen = state["gen"]
    selected = state["selected"]
    parts = []
    for row in tokens:
        cls = "s2-tok"
        if row["eos"]:
            cls += " s2-eos"
        if row["i"] == selected:
            cls += " s2-on"
        text = html.escape(_visible(row["piece"]))
        parts.append(
            f"<span class='{cls}' data-gen='{gen}' data-i='{row['i']}'>"
            f"{text}</span>"
        )
    return "<div class='s2-tokens'>" + "".join(parts) + "</div>"


def _show_tokens():
    token_html.value = _token_markup()


def _point_text(name, xy):
    x, y = xy
    extra = "" if _on_board(x, y) else " (off the board)"
    return f"{name}=({x:.4f}, {y:.4f}){extra}"


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def _say(text):
    log.append_stdout(text if text.endswith("\n") else text + "\n")


def _set_busy(flag):
    state["busy"] = flag
    for widget in controls:
        widget.disabled = flag
    settings.set_disabled(flag)


def _fresh_frame():
    state["frame"] = canonical_frame(state["game"])
    _show_board()


def _start_game():
    state["game"] = new_multi_gold_game(n_gold=None, opening="any")
    state["selected"] = None
    tokens.clear()
    state["gen"] += 1
    _fresh_frame()
    _show_tokens()


_FRAME_PATH = os.path.join(tempfile.gettempdir(), "testing_s2_frame.png")


def _write_frame(frame):
    Image.fromarray(np.ascontiguousarray(frame), mode="RGB").save(_FRAME_PATH)
    return _FRAME_PATH


def _messages(mode, frame, question):
    """Prompt text from the trainer's builders.

    ``start`` is ``beginning_messages`` with an empty notepad. ``trace``
    is ``synthetic_prompt`` on this board, with the question box used
    verbatim. Returns messages plus the fake-target fields (None on start).
    """
    path = _write_frame(frame)
    if mode == "start":
        return beginning_messages(path, format_notepad([])), None, None
    if mode == "trace":
        built = synthetic_prompt(
            game_to_settings_dict(state["game"]),
            random.Random(),
            path,
            question=question,
        )
        return built.messages, built.target_note, built.question_kind
    raise RuntimeError(f"unknown prompt mode {mode!r}")


def _log_prompt(messages, target_note=None, question_kind=None):
    if question_kind is not None:
        _say(f"target_note={target_note!r} question_kind={question_kind}")
    for message in messages:
        role = message.get("role", "?")
        content = message.get("content")
        chunks = []
        if isinstance(content, str):
            chunks.append(content)
        elif isinstance(content, list):
            for part in content:
                if not isinstance(part, dict):
                    continue
                if part.get("type") == "image":
                    chunks.append("[image: current frame]")
                elif part.get("type") == "text":
                    chunks.append(part.get("text") or "")
        _say(f"--- {role} ---\n" + "\n".join(chunks))


def _record_token(tokenizer, token_id, coords, eos):
    vals = [float(v) for v in coords.detach().float().cpu().tolist()]
    if len(vals) != 6:
        raise RuntimeError(
            f"coordinate head returned {len(vals)} values, expected 6"
        )
    sx, sy, vx, vy, bx, by = vals
    piece = tokenizer.decode([token_id], skip_special_tokens=False)
    tokens.append({
        "i": len(tokens),
        "id": token_id,
        "piece": piece,
        "eos": token_id in eos,
        "s": (sx, sy),
        "v": (vx, vy),
        "v_bar": (bx, by),
        "avg": ((vx + bx) / 2.0, (vy + by) / 2.0),
    })


def _generate(mode, question):
    net = state["s2"]
    net.vl.model.eval()
    frame = canonical_frame(state["game"])
    state["frame"] = frame
    messages, note, qkind = _messages(mode, frame, question)
    _log_prompt(messages, note, qkind)
    encoded = net.vl.encode_messages(messages)
    prompt_len = int(encoded["input_ids"].shape[1])
    _say(f"prompt tokens {prompt_len}")
    sampling = net.vl._sampling_kwargs()
    eos = _eos_ids(net.vl)
    tokenizer = getattr(net.vl.processor, "tokenizer", net.vl.processor)
    _sync()
    started = time.perf_counter()
    logits, coords, past = net.prefill(encoded)
    seq_len = prompt_len
    for token_index in range(BEGINNING_MAX_NEW_TOKENS):
        token_ids = _sample(logits, sampling)
        token_id = int(token_ids[0].item())
        _record_token(tokenizer, token_id, coords[0], eos)
        if token_id in eos or token_index + 1 == BEGINNING_MAX_NEW_TOKENS:
            break
        logits, coords, past = net.decode(token_ids, past, seq_len)
        seq_len += 1
        if token_index % 16 == 0:
            status.value = f"Generating token {token_index + 1}."
    _sync()
    elapsed = time.perf_counter() - started
    ids = [row["id"] for row in tokens]
    text = tokenizer.decode(ids, skip_special_tokens=True).strip()
    if tokens and tokens[-1]["eos"]:
        why = "eos"
    else:
        why = f"cap {BEGINNING_MAX_NEW_TOKENS}"
    _say(f"{len(tokens)} tokens in {elapsed:.1f}s ({why})")
    if text:
        _say(text)
    _say("")


_opts = _snap_options()
_questions = _question_options()
ckpt = widgets.Dropdown(
    options=_opts,
    value=_opts[0][1],
    description="Snapshot:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="640px"),
)
refresh_btn = widgets.Button(description="Refresh")
load_btn = widgets.Button(description="Load", button_style="primary")
new_btn = widgets.Button(description="New board")
clear_btn = widgets.Button(description="Clear")
standard = widgets.Dropdown(
    options=_questions,
    value=S2_BEGINNING_USER,
    description="Standard:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="100%"),
)
question = widgets.Textarea(
    value=S2_BEGINNING_USER,
    description="Question:",
    continuous_update=True,
    style={"description_width": "90px"},
    layout=widgets.Layout(width="100%", height="72px"),
)
prompt_mode = widgets.ToggleButtons(
    options=[
        ("prompt at start of game", "start"),
        ("prompt with fake NAMS trace and target selected", "trace"),
    ],
    value="start",
    description="Prompt:",
    style={"description_width": "90px", "button_width": "480px"},
    layout=widgets.Layout(width="100%"),
)
generate_btn = widgets.Button(description="Generate", button_style="primary")
show_all = widgets.Checkbox(value=False, description="show all", indent=False)
legend = widgets.HTML(
    "<span style='color:#1e5aff'>● s</span> &nbsp; "
    "<span style='color:#ff0000'>● v</span> &nbsp; "
    "<span style='color:#ff8c00'>● v_bar</span> &nbsp; "
    "<span style='color:#00c846'>● average of v and v_bar</span>"
)
readout = widgets.HTML("Click a token after Generate.")
status = widgets.HTML("Load a snapshot, then Generate.")
board = widgets.Image(format="png", width=BOARD_PX, height=BOARD_PX)
token_html = widgets.HTML()
log = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc",
    height="280px",
    max_height="280px",
    overflow_y="auto",
    width="100%",
    flex="0 0 auto",
))
log.add_class("s2-log")
pick_box = widgets.Text(value="")
pick_box.add_class("s2-pick")
pick_box.layout.height = "0px"
pick_box.layout.visibility = "hidden"
pick_box.layout.overflow = "hidden"
settings = settings_editor(height_px=560)
controls = (
    ckpt, refresh_btn, load_btn, new_btn, clear_btn,
    standard, question, prompt_mode, generate_btn, show_all,
)


def on_refresh(_):
    selected = ckpt.value
    options = _snap_options()
    ckpt.options = options
    values = [value for _, value in options]
    ckpt.value = selected if selected in values else values[0]
    n = sum(1 for value in values if value)
    _say(f"{n} snapshot(s) under weights/localizer and weights/s2")


def on_load(_):
    path = ckpt.value
    if not path:
        _say("no snapshot selected")
        return
    if state["busy"]:
        return
    _set_busy(True)
    status.value = "Loading snapshot."
    try:
        if state["s2"] is None:
            state["s2"] = GemmaS2()
        net = state["s2"]
        net.load_snapshot(path)
        n_head = sum(p.numel() for p in net.localizer.parameters())
        device = next(net.vl.model.parameters()).device
        _say(f"loaded {path}  localizer {n_head} params  on {device}")
        status.value = "Edit the question if you want, then Generate."
    except Exception:
        status.value = "Load failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_clear(_):
    if state["busy"]:
        return
    tokens.clear()
    state["selected"] = None
    state["gen"] += 1
    log.clear_output()
    readout.value = "Click a token after Generate."
    _show_tokens()
    _show_board()
    status.value = "Record cleared."


def on_new(_):
    if state["busy"]:
        return
    _start_game()
    d = game_to_settings_dict(state["game"])
    settings.show_view(d)
    readout.value = "Click a token after Generate."
    _say(
        f"new board  gold={len(d.get('gold') or [])}  "
        f"openings={len(d.get('openings') or [])}"
    )
    if state["s2"] is None or state["s2"].vl.model is None:
        status.value = "Load a snapshot, then Generate."
    else:
        status.value = "Edit the question if you want, then Generate."


def on_settings_edit(_):
    if state["busy"]:
        return
    settings.enter_edit(game_to_settings_dict(state["game"]))


def on_settings_render(_):
    if state["busy"] or not settings.is_editing():
        return
    try:
        state["game"] = apply_edited_settings_dict(settings.collect())
    except ValueError as exc:
        settings.set_error(f"Error, fix settings before rendering.\n{exc}")
        return
    tokens.clear()
    state["selected"] = None
    state["gen"] += 1
    readout.value = "Click a token after Generate."
    _fresh_frame()
    _show_tokens()
    settings.show_view(game_to_settings_dict(state["game"]))
    _say("rendered settings")


def on_standard(change):
    chosen = change["new"]
    if not chosen:
        return
    question.unobserve(on_question_edit, names="value")
    try:
        question.value = chosen
    finally:
        question.observe(on_question_edit, names="value")


def on_question_edit(change):
    if standard.value and change["new"] != standard.value:
        standard.unobserve(on_standard, names="value")
        try:
            standard.value = ""
        finally:
            standard.observe(on_standard, names="value")


def on_show_all(_change):
    _show_board()


def on_generate(_):
    if state["busy"]:
        return
    if settings.is_editing():
        _say("Render game settings before Generate.")
        return
    if state["s2"] is None or state["s2"].vl.model is None:
        _say("no snapshot loaded")
        return
    question_text = question.value
    if not question_text.strip():
        _say("question is empty")
        status.value = "The question box is empty."
        return
    mode = prompt_mode.value
    _set_busy(True)
    status.value = "Generating."
    tokens.clear()
    state["selected"] = None
    state["gen"] += 1
    readout.value = "Click a token after Generate."
    _show_tokens()
    _show_board()
    try:
        _generate(mode, question_text)
    except Exception:
        status.value = "Generate failed."
        _say(traceback.format_exc())
        raise
    finally:
        _show_tokens()
        _show_board()
        _set_busy(False)
    status.value = (
        "Click a token. Blue is s, red is v, orange is v_bar, "
        "green is the average of v and v_bar."
    )


def on_pick(change):
    raw = change["new"]
    if not raw or raw.count(",") != 2:
        return
    seq_s, gen_s, idx_s = raw.split(",")
    try:
        seq = int(seq_s)
        gen = int(gen_s)
        index = int(idx_s)
    except ValueError:
        return
    if seq <= state["ignore_through"] or state["busy"]:
        return
    if gen != state["gen"]:
        return
    if index < 0 or index >= len(tokens):
        _say(f"token {index} is not in this reply ({len(tokens)} tokens)")
        return
    state["selected"] = index
    row = tokens[index]
    readout.value = (
        f"token {index} &nbsp; {html.escape(row['piece'])}<br>"
        f"{_point_text('s', row['s'])} &nbsp; "
        f"{_point_text('v', row['v'])}<br>"
        f"{_point_text('v_bar', row['v_bar'])} &nbsp; "
        f"{_point_text('avg', row['avg'])}"
    )
    _show_tokens()
    _show_board()
    _say(
        f"token {index} {row['piece']!r}  "
        f"s={row['s'][0]:.4f},{row['s'][1]:.4f}  "
        f"v={row['v'][0]:.4f},{row['v'][1]:.4f}  "
        f"v_bar={row['v_bar'][0]:.4f},{row['v_bar'][1]:.4f}  "
        f"avg={row['avg'][0]:.4f},{row['avg'][1]:.4f}"
    )
    state["ignore_through"] = seq


refresh_btn.on_click(on_refresh)
load_btn.on_click(on_load)
new_btn.on_click(on_new)
clear_btn.on_click(on_clear)
generate_btn.on_click(on_generate)
show_all.observe(on_show_all, names="value")
standard.observe(on_standard, names="value")
question.observe(on_question_edit, names="value")
settings.edit_btn.on_click(on_settings_edit)
settings.render_btn.on_click(on_settings_render)
pick_box.observe(on_pick, names="value")

_start_game()
settings.show_view(game_to_settings_dict(state["game"]))
_show_tokens()
_say(f"cuda {torch.cuda.is_available()}")
n_snap = sum(1 for _, value in _opts if value)
_say(f"{n_snap} snapshot(s) under weights/localizer and weights/s2")
_say(
    "Generate runs one reply. A standard question only fills the box. "
    "Click a token to plot s / v / v_bar / their average."
)

display(HTML(
    """
<style>
.s2-tokens {
  white-space: pre-wrap;
  line-height: 1.8;
  border: 1px solid #ccc;
  padding: 8px;
  max-height: 240px;
  overflow-y: auto;
  font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
}
.s2-tok { cursor: pointer; border-radius: 3px; }
.s2-tok:hover { background: #e8eefc; }
.s2-tok.s2-on { background: #ffe08a; }
.s2-eos { opacity: 0.55; }
.s2-log {
  height: 280px !important;
  max-height: 280px !important;
  overflow-y: auto !important;
  flex: 0 0 auto !important;
}
</style>
<script>
(function () {
  if (window.__s2TokenClick) { return; }
  window.__s2TokenClick = true;
  window.addEventListener("click", function (ev) {
    var span = ev.target && ev.target.closest && ev.target.closest(".s2-tok");
    if (!span) { return; }
    var box = document.querySelector(".s2-pick input");
    if (!box) { return; }
    window.__s2PickN = (window.__s2PickN || 0) + 1;
    var gen = span.getAttribute("data-gen") || "0";
    var i = span.getAttribute("data-i");
    box.value = window.__s2PickN + "," + gen + "," + i;
    box.dispatchEvent(new Event("input", { bubbles: true }));
  }, true);
  if (!window.__s2LogFollow) {
    window.__s2LogFollow = true;
    setInterval(function () {
      var box = document.querySelector(".s2-log");
      if (!box) { return; }
      var gap = box.scrollHeight - box.scrollTop - box.clientHeight;
      if (gap < 80) { box.scrollTop = box.scrollHeight; }
    }, 250);
  }
})();
</script>
"""
))
display(widgets.VBox([
    widgets.HBox([ckpt, refresh_btn, load_btn, new_btn, clear_btn]),
    standard,
    question,
    prompt_mode,
    widgets.HBox([generate_btn, show_all, legend]),
    readout,
    widgets.HBox([
        widgets.VBox(
            [status, board],
            layout=widgets.Layout(flex="0 0 auto"),
        ),
        widgets.VBox(
            [settings.box],
            layout=widgets.Layout(flex="1 0 520px", min_width="520px"),
        ),
    ], layout=widgets.Layout(width="100%", align_items="flex-start")),
    token_html,
    log,
    pick_box,
]))
